# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR^2 dataset package](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) using the `mlcroissant` library. It follows the Croissant metadata schema and shows stepwise how to reveal the dataset's structure, load its data, perform EDA, and visualize insights.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review the available record sets, their fields, and their `@id`s.

For each record set in the dataset, we will list its `@id`, name, and for each field, print its `@id`, name, and data type.

In [ ]:
print("Available record sets and their fields:")

record_sets = []  # Collect record set IDs for later use
rs_name_map = {}  # Maps record set @id to a readable name
field_map = {}    # Maps record set @id to a list of field @ids

for record_set in metadata.record_sets:
    rset_id = record_set.id
    record_sets.append(rset_id)
    rs_name_map[rset_id] = getattr(record_set, 'name', rset_id)
    print(f"\nRecord Set: {rset_id} | Name: {getattr(record_set, 'name', rset_id)}")
    field_map[rset_id] = []
    for field in record_set.fields:
        field_id = field.id
        field_map[rset_id].append(field_id)
        print(f"  Field: {field_id}")
        print(f"    Name: {getattr(field, 'name', field_id)}")
        print(f"    Data type: {getattr(field, 'data_type', 'Unknown')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Below, we extract all record sets and create a Pandas DataFrame for each.

Refer to record sets and fields using their `@id` values as shown in the overview above.

In [ ]:
# Extract data from each record set into dataframes
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set {record_set_id}.")
    else:
        print(f"No records found for record set {record_set_id}.")

# For demonstration, use the first non-empty record set (if any)
main_record_set_id = None
for rsid in record_sets:
    if rsid in dataframes:
        main_record_set_id = rsid
        break

if main_record_set_id:
    print(f"\nExample record set: {main_record_set_id}")
    print(f"Columns: {dataframes[main_record_set_id].columns.tolist()}")
    display(dataframes[main_record_set_id].head())
else:
    print("No record set contains data.")

## 4. Exploratory Data Analysis (EDA)

We will select a numeric field from the main record set, filter records, normalize the field, and group data by another available field.

**All field and record set selections below use their `@id` as referenced above.**

In [ ]:
# EDA: Example with a numeric field

# If data is available, pick a likely numeric field (edit as needed for your schema):
if main_record_set_id:
    df = dataframes[main_record_set_id]

    # Try to automatically detect numeric fields from the record set
    numeric_field_id = None
    group_field_id = None
    # Find the first numeric-type field according to Croissant metadata
    for record_set in metadata.record_sets:
        if record_set.id == main_record_set_id:
            for field in record_set.fields:
                dtype = str(getattr(field, 'data_type', '')).lower()
                if dtype in ['number', 'integer', 'float'] and field.id in df.columns:
                    numeric_field_id = field.id
                    break
            # If possible, also select a group-by categorical field
            for field in record_set.fields:
                dtype = str(getattr(field, 'data_type', '')).lower()
                if dtype in ['string', 'text', 'category'] and field.id in df.columns:
                    group_field_id = field.id
                    break
            break

    if numeric_field_id is not None:
        # Try to convert to numeric
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')

        threshold = df[numeric_field_id].quantile(0.75) if not df[numeric_field_id].isnull().all() else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        mean = filtered_df[numeric_field_id].mean()
        std = filtered_df[numeric_field_id].std()
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean) / std if std else 0
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        if group_field_id is not None and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped data by {group_field_id}:")
            display(grouped_df)
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric field detected in the main record set.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and relationship with the grouped field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(data=dataframes[main_record_set_id], x=numeric_field_id, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.tight_layout()
    plt.show()

    # If group field available, boxplot
    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(10, 5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=20)
        plt.tight_layout()
        plt.show()

## 6. Conclusion

This notebook demonstrated:
- Loading FAIR^2 dataset metadata and records using `mlcroissant`.
- Exploring dataset structure and accessing fields via their `@id`s.
- Extracting record set content for tabular analysis using Pandas.
- Performing simple EDA using a detected numeric field and optionally grouping by a categorical field.
- Visualizing distributions using Matplotlib and Seaborn.

For in-depth biomedical or clinicopathological analysis, consult the dataset documentation and use precise `@id`s and field semantics as shown above. Review potential variable meanings (MSI status, anatomical location, comorbidities, etc.) and handle sensitive medical data per your institution's policy.